In [ ]:
## FEITO POR Gustavo Emanuel

import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

device

device(type='cuda')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_data = datasets.ImageFolder("drive/MyDrive/dataset/Training", transform=transform)
test_data  = datasets.ImageFolder("drive/MyDrive/dataset/Testing", transform=transform)

train_loader = torch.utils.data.DataLoader(
    train_data,
    batch_size=256,           # Aumente se couber na memória
    shuffle=True,
    num_workers=4,           # Pode testar com 2, 4, 8...
    pin_memory=True
)

test_loader  = DataLoader(test_data, batch_size=32)

torch.backends.cudnn.benchmark = True


## SIMPLE CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        self.conv_layer = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3),  # Entrada RGB
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        dummy_input = torch.randn(1, 3, 128, 128) # Assuming input image size is 150x150
        dummy_output = self.conv_layer(dummy_input)
        fc_input_size = dummy_output.shape[1] * dummy_output.shape[2] * dummy_output.shape[3]

        self.fc_layer = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)  # 4 classes
        )

    def forward(self, x):
        x = self.conv_layer(x)
        x = self.fc_layer(x)
        return x

model = SimpleCNN().to(device)


In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)


def train_model(epochs):
    for epoch in range(epochs):
        model.train()
        running_loss = 0
        correct, total = 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()


            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

        acc = 100 * correct / total
        print(f"Epoch {epoch+1}, Loss: {running_loss:.4f}, Accuracy: {acc:.2f}%")

#train_model(epochs=15)


In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.numpy())

# Relatório com métricas
classes = test_data.classes
print(classification_report(y_true, y_pred, target_names=classes))

              precision    recall  f1-score   support

      glioma       0.97      0.96      0.97       300
  meningioma       0.96      0.95      0.96       306
     notumor       0.99      1.00      0.99       405
   pituitary       0.98      1.00      0.99       300

    accuracy                           0.98      1311
   macro avg       0.98      0.98      0.98      1311
weighted avg       0.98      0.98      0.98      1311



## CNN VGG

In [ ]:
class CNN_VGG(nn.Module):
    def __init__(self):
        super(CNN_VGG, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # [32, 75, 75]

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # [64, 37, 37]

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2),  # [128, 18, 18]
        )

        dummy_input = torch.randn(1, 3, 128, 128) # Use the actual input size
        dummy_output = self.conv_layers(dummy_input)
        fc_input_size = dummy_output.shape[1] * dummy_output.shape[2] * dummy_output.shape[3]

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_input_size, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 4)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = CNN_VGG().to(device)

train_model(epochs=500)


Epoch 1, Loss: 33.5107, Accuracy: 24.12%
Epoch 2, Loss: 33.4825, Accuracy: 24.23%
Epoch 3, Loss: 33.4691, Accuracy: 24.02%
Epoch 4, Loss: 33.3983, Accuracy: 24.21%
Epoch 5, Loss: 33.4928, Accuracy: 24.44%
Epoch 6, Loss: 33.4934, Accuracy: 23.79%
Epoch 7, Loss: 33.3808, Accuracy: 24.51%
Epoch 8, Loss: 33.5105, Accuracy: 24.28%
Epoch 9, Loss: 33.5138, Accuracy: 24.37%
Epoch 10, Loss: 33.4427, Accuracy: 24.25%
Epoch 11, Loss: 33.5819, Accuracy: 23.83%
Epoch 12, Loss: 33.4877, Accuracy: 24.40%
Epoch 13, Loss: 33.4181, Accuracy: 23.76%
Epoch 14, Loss: 33.5186, Accuracy: 24.05%
Epoch 15, Loss: 33.5567, Accuracy: 23.32%
Epoch 16, Loss: 33.3838, Accuracy: 25.18%
Epoch 17, Loss: 33.4261, Accuracy: 23.88%
Epoch 18, Loss: 33.3765, Accuracy: 24.67%
Epoch 19, Loss: 33.5374, Accuracy: 24.18%
Epoch 20, Loss: 33.5076, Accuracy: 24.63%
Epoch 21, Loss: 33.4136, Accuracy: 24.16%
Epoch 22, Loss: 33.4920, Accuracy: 23.72%
Epoch 23, Loss: 33.5346, Accuracy: 24.04%
Epoch 24, Loss: 33.5161, Accuracy: 24.56%
E

In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.numpy())

# Relatório com métricas
classes = test_data.classes
print(classification_report(y_true, y_pred, target_names=classes))

              precision    recall  f1-score   support

      glioma       0.11      0.12      0.12       300
  meningioma       0.28      0.15      0.19       306
     notumor       0.26      0.48      0.34       405
   pituitary       0.23      0.06      0.09       300

    accuracy                           0.22      1311
   macro avg       0.22      0.20      0.19      1311
weighted avg       0.22      0.22      0.20      1311



## CNN_LeNet

In [ ]:
import torch.nn as nn

class CNN_LeNet(nn.Module):
    def __init__(self):
        super(CNN_LeNet, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5),     # [3, 150, 150] → [32, 146, 146]
            nn.ReLU(),
            nn.MaxPool2d(2),                     # [32, 73, 73]

            nn.Conv2d(32, 64, kernel_size=5),    # [64, 69, 69]
            nn.ReLU(),
            nn.MaxPool2d(2),                     # [64, 34, 34]
        )

        dummy_input = torch.randn(1, 3, 128, 128) # Use the actual input size
        dummy_output = self.conv_layers(dummy_input)
        fc_input_size = dummy_output.shape[1] * dummy_output.shape[2] * dummy_output.shape[3]

        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(fc_input_size, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, 4)  # 4 classes
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = self.fc_layers(x)
        return x

model = CNN_LeNet().to(device)

train_model(epochs=15)

Epoch 1, Loss: 248.6400, Accuracy: 24.46%
Epoch 2, Loss: 248.5534, Accuracy: 24.96%
Epoch 3, Loss: 248.6139, Accuracy: 24.82%
Epoch 4, Loss: 248.6921, Accuracy: 23.84%
Epoch 5, Loss: 248.5582, Accuracy: 25.21%
Epoch 6, Loss: 248.6553, Accuracy: 24.63%
Epoch 7, Loss: 248.5810, Accuracy: 24.60%
Epoch 8, Loss: 248.6426, Accuracy: 24.96%
Epoch 9, Loss: 248.6225, Accuracy: 24.61%
Epoch 10, Loss: 248.5788, Accuracy: 24.23%
Epoch 11, Loss: 248.5190, Accuracy: 24.79%
Epoch 12, Loss: 248.6752, Accuracy: 23.27%
Epoch 13, Loss: 248.6607, Accuracy: 24.58%
Epoch 14, Loss: 248.5865, Accuracy: 24.79%
Epoch 15, Loss: 248.4734, Accuracy: 25.07%


In [ ]:
model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.numpy())

# Relatório com métricas
classes = test_data.classes
print(classification_report(y_true, y_pred, target_names=classes))

              precision    recall  f1-score   support

      glioma       0.24      0.96      0.39       300
  meningioma       0.35      0.02      0.04       306
     notumor       0.00      0.00      0.00       405
   pituitary       0.18      0.06      0.09       300

    accuracy                           0.24      1311
   macro avg       0.19      0.26      0.13      1311
weighted avg       0.18      0.24      0.12      1311



/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.11/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
